# LazyQSAR Classifier Benchmark

This notebook is a lightweight analysis layer over `benchmarks/run_benchmark.py`.
It reuses the shared benchmark helpers so the notebook and CLI stay in sync.

It compares:

| Model | Input | Notes |
|---|---|---|
| `LazyClassifier` | Morgan fingerprints | Descriptor-agnostic API |
| `LazyClassifier (ONNX)` | Morgan fingerprints | Save/load inference roundtrip |
| `LazyClassifierQSAR` | SMILES | Built-in descriptor pipeline (`mode="fast"`) |
| `LR (default)` | Morgan fingerprints | sklearn baseline |
| `XGB (default)` | Morgan fingerprints | xgboost baseline |
| `RF (default)` | Morgan fingerprints | sklearn baseline |

Core metrics: ROC-AUC, PR-AUC, calibration (ECE), fit time, and inference time.


In [ ]:
import importlib.util
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Image, display

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-whitegrid")

print("Notebook environment ready")

In [ ]:
# Configuration
# Set DATASET_NAMES = None to run the full benchmark suite.
# Set MODE = "agnostic", "qsar", or "both".

MODE = "both"
DATASET_NAMES = None
SHOW_PER_DATASET_CURVES = True

cwd = Path.cwd().resolve()
if (cwd / "benchmarks" / "run_benchmark.py").exists():
    REPO_ROOT = cwd
elif (cwd / "run_benchmark.py").exists() and cwd.name == "benchmarks":
    REPO_ROOT = cwd.parent
else:
    raise FileNotFoundError(
        "Run this notebook from the repository root or the benchmarks/ directory."
    )

BENCHMARKS_DIR = REPO_ROOT / "benchmarks"
RESULTS_DIR = BENCHMARKS_DIR / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

module_path = BENCHMARKS_DIR / "run_benchmark.py"
spec = importlib.util.spec_from_file_location("lazyqsar_benchmark", module_path)
rb = importlib.util.module_from_spec(spec)
spec.loader.exec_module(rb)

DATA_DIR = rb._DEFAULT_DATA_DIR
SELECTED_DATASETS = DATASET_NAMES or rb.DATASETS

print(f"REPO_ROOT: {REPO_ROOT}")
print(f"DATA_DIR: {DATA_DIR}")
print(f"RESULTS_DIR: {RESULTS_DIR}")
print(f"MODE: {MODE}")
print(f"Datasets requested: {len(SELECTED_DATASETS)}")

In [ ]:
if not DATA_DIR.exists():
    raise FileNotFoundError(
        f"Data directory not found: {DATA_DIR}\\n"
        "Update DATA_DIR in the configuration cell before running the benchmark."
    )

available_tabs = sorted(DATA_DIR.glob("*.tab"))
available_names = {path.stem for path in available_tabs}
missing = [name for name in SELECTED_DATASETS if name not in available_names]
ready = [name for name in SELECTED_DATASETS if name in available_names]

summary = pd.DataFrame({"requested": pd.Series(SELECTED_DATASETS, dtype="object")})
summary["available"] = summary["requested"].isin(available_names)

print(f"Found {len(available_tabs)} dataset files")
print(f"Ready to run: {len(ready)}")
if missing:
    print("Missing datasets:", ", ".join(missing))

summary

## Run Benchmark

The next cell executes the same dataset-level benchmark routine used by the CLI script,
then keeps the results in memory for interactive analysis.


In [ ]:
all_results = []
all_curves = []

for idx, name in enumerate(ready):
    path = DATA_DIR / f"{name}.tab"
    smiles, y = rb.load_dataset(path)
    print()
    print("-" * 70)
    print(f"{name}: n={len(y):,}  pos={y.mean():.1%}  neg={1 - y.mean():.1%}")
    print("-" * 70)

    results, curves = rb.run_dataset(
        name=name,
        smiles=smiles,
        y=y,
        mode=MODE,
        output_dir=str(RESULTS_DIR),
        is_first_dataset=(idx == 0),
    )
    all_results.extend(results)
    all_curves.extend(curves)

    for row in results:
        if isinstance(row.get("error"), str):
            print(f"  {row['model']:35s} ERROR  {row['error']}")
            continue
        fit = row.get("fit_time")
        infer = row.get("infer_time")
        fit_txt = "-" if pd.isna(fit) else f"{fit:.3f}s"
        infer_txt = "-" if pd.isna(infer) else f"{infer:.4f}s"
        print(
            f"  {row['model']:35s} "
            f"roc={row['roc_auc']:.4f} "
            f"pr={row['pr_auc']:.4f} "
            f"ece={row['ece']:.4f} "
            f"fit={fit_txt} "
            f"infer={infer_txt}"
        )

if not all_results:
    raise RuntimeError("No benchmark results were collected.")

print()
print(f"Collected {len(all_results)} result rows and {len(all_curves)} curve bundles.")

In [ ]:
df = pd.DataFrame(all_results)
present_models = [m for m in rb.MODEL_ORDER if m in df["model"].unique()]

pivot_roc = rb.build_pivot(all_results, "roc_auc")
pivot_pr = rb.build_pivot(all_results, "pr_auc")
pivot_ece = rb.build_pivot(all_results, "ece")
pivot_fit = rb.build_pivot(all_results, "fit_time")
pivot_infer = rb.build_pivot(all_results, "infer_time")
mean_ranks = rb.mean_rank_table(pivot_roc)

aggregate = (
    df.groupby("model", dropna=False)
    .agg(
        mean_roc_auc=("roc_auc", "mean"),
        mean_pr_auc=("pr_auc", "mean"),
        mean_ece=("ece", "mean"),
        mean_fit_time=("fit_time", "mean"),
        mean_infer_time=("infer_time", "mean"),
    )
    .reindex(present_models)
)
aggregate["mean_rank_roc_auc"] = mean_ranks.reindex(aggregate.index)
aggregate = aggregate.sort_values(
    ["mean_rank_roc_auc", "mean_roc_auc"], ascending=[True, False]
)

aggregate

## Summary Tables


In [ ]:
(
    aggregate.style.background_gradient(
        subset=["mean_roc_auc", "mean_pr_auc"], cmap="RdYlGn"
    )
    .background_gradient(
        subset=["mean_ece", "mean_fit_time", "mean_infer_time", "mean_rank_roc_auc"],
        cmap="RdYlGn_r",
    )
    .format(
        {
            "mean_roc_auc": "{:.4f}",
            "mean_pr_auc": "{:.4f}",
            "mean_ece": "{:.4f}",
            "mean_fit_time": "{:.3f}s",
            "mean_infer_time": "{:.4f}s",
            "mean_rank_roc_auc": "{:.2f}",
        }
    )
    .set_caption("Aggregate benchmark summary")
)

In [ ]:
(
    pivot_roc.style.background_gradient(cmap="RdYlGn", axis=1, vmin=0.5, vmax=1.0)
    .format("{:.4f}")
    .set_caption("ROC-AUC by dataset")
)

In [ ]:
(
    pivot_pr.style.background_gradient(cmap="RdYlGn", axis=1)
    .format("{:.4f}")
    .set_caption("PR-AUC by dataset")
)

In [ ]:
(
    pivot_ece.style.background_gradient(cmap="RdYlGn_r", axis=1)
    .format("{:.4f}")
    .set_caption("Expected calibration error by dataset (lower is better)")
)

In [ ]:
pd.concat({"fit_time_s": pivot_fit, "infer_time_s": pivot_infer}, axis=1)

## Plots

The next cell regenerates the same report plots used by the CLI benchmark and shows
those images inline so the notebook stays compact.


In [ ]:
plot_paths = {
    "roc_bar": RESULTS_DIR / "benchmark_bars_auroc.png",
    "pr_bar": RESULTS_DIR / "benchmark_bars_aupr.png",
    "timing": RESULTS_DIR / "benchmark_timing.png",
    "radar": RESULTS_DIR / "benchmark_radar.png",
    "rank": RESULTS_DIR / "benchmark_rank_scatter.png",
}

rb.plot_bars(pivot_roc, plot_paths["roc_bar"], "ROC-AUC by dataset")
rb.plot_bars(pivot_pr, plot_paths["pr_bar"], "PR-AUC by dataset")
rb.plot_timing_bars(all_results, plot_paths["timing"])
rb.plot_radar(all_results, plot_paths["radar"])
rb.plot_rank_scatter(pivot_roc, plot_paths["rank"])

for key in ["roc_bar", "pr_bar", "timing", "radar", "rank"]:
    print(f"Saved {key}: {plot_paths[key]}")
    display(Image(filename=str(plot_paths[key])))

In [ ]:
if SHOW_PER_DATASET_CURVES and all_curves:
    detail_paths = {
        "roc_curves": RESULTS_DIR / "roc_curves.png",
        "pr_curves": RESULTS_DIR / "pr_curves.png",
        "calibration_curves": RESULTS_DIR / "calibration_curves.png",
    }
    rb.plot_roc_curves(all_curves, detail_paths["roc_curves"])
    rb.plot_pr_curves(all_curves, detail_paths["pr_curves"])
    rb.plot_calibration_curves(all_curves, detail_paths["calibration_curves"])

    for key, path in detail_paths.items():
        print(f"Saved {key}: {path}")
        display(Image(filename=str(path)))
else:
    print("Skipping per-dataset ROC/PR/calibration curve plots.")

In [ ]:
csv_path = RESULTS_DIR / "benchmark_results.csv"
df.to_csv(csv_path, index=False)
print(f"Saved results CSV: {csv_path}")
print(f"Artifacts directory: {RESULTS_DIR}")